# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de pedidos WMS

Este notebook trabaja **solo** con `andinalog_wms_orders.csv`. Lee la capa Bronze desde `datasets/AndinaLog_03B_Bronce/`, detecta problemas y conserva los catorce campos originales. No normaliza identificadores, no convierte fechas, no imputa cantidades, no corrige tiempos ni recalcula indicadores OTIF. Esas decisiones corresponden al notebook 2 después de acordar las reglas.

Cada ejecución reemplaza cuatro archivos en `S4/andinalog_wms_orders/notebook1/salidas/`:

1. `andinalog_wms_orders_diagnosticado.csv`: todas las filas, los campos originales y solo `fila_bronze`, `columnas_con_problemas` y `en_cuarentena`.
2. `andinalog_wms_orders_problemas.csv`: una fila por problema, con columna, código estable y evidencia.
3. `andinalog_wms_orders_cuarentena.csv`: extracto informativo de las filas marcadas.
4. `andinalog_wms_orders_reporte_calidad.csv`: conteos y huella SHA-256 del CSV de origen.


## 1 · Configuración y origen

`ENTORNO = "auto"` usa Drive cuando se ejecuta en Google Colab y busca la raíz del repositorio cuando se ejecuta localmente. En Colab, ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`.


In [ ]:
from pathlib import Path
import hashlib
import os
import re
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_wms_orders.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-WMS-orders-diagnostico-v1"

COLUMNAS_ORIGINALES = [
    "order_id", "cliente_id", "producto_id", "fecha_despacho", "centro_distribucion",
    "camion_id", "chofer_id", "cantidad_solicitada", "cantidad_entregada",
    "tiempo_entrega_prometido_hrs", "tiempo_entrega_real_hrs",
    "otif_on_time", "otif_in_full", "otif",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "S4" / "andinalog_wms_orders" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales y no sustituyen los valores originales.


In [ ]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


## 3 · Catálogo y reglas de diagnóstico

Los códigos son estables para que el notebook 2 pueda identificar el problema exacto. Las reglas comprueban estructura, dominio y coherencia interna del pedido. No consultan otros CSV ni deciden cómo corregir un hallazgo.


In [ ]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("order_id", "FALTANTE", "Identificador vacío"),
    ("order_id", "FORMATO_INVALIDO", "No cumple ORD-2026-##### exactamente"),
    ("cliente_id", "FALTANTE", "Identificador vacío"),
    ("cliente_id", "FORMATO_INVALIDO", "No cumple CLI-### exactamente"),
    ("producto_id", "FALTANTE", "Identificador vacío"),
    ("producto_id", "FORMATO_INVALIDO", "No cumple PROD-### exactamente"),
    ("camion_id", "FALTANTE", "Identificador vacío"),
    ("camion_id", "FORMATO_INVALIDO", "No cumple CAM-## exactamente"),
    ("chofer_id", "FALTANTE", "Identificador vacío"),
    ("chofer_id", "FORMATO_INVALIDO", "No cumple CHO-### exactamente"),
    ("order_id", "DUPLICADO", "Identificador repetido; se marca la aparición posterior"),
    ("fecha_despacho", "FECHA_INVALIDA", "No cumple AAAA-MM-DD HH:MM:SS o no existe en el calendario"),
    ("centro_distribucion", "FALTANTE", "Centro vacío"),
    ("centro_distribucion", "VALOR_NO_RECONOCIDO", "Centro fuera del dominio operativo declarado"),
    ("cantidad_solicitada", "FALTANTE", "Cantidad vacía"),
    ("cantidad_solicitada", "NO_NUMERICA", "Valor no convertible a número"),
    ("cantidad_solicitada", "NO_POSITIVA", "Cantidad menor o igual que cero"),
    ("cantidad_entregada", "FALTANTE", "Cantidad vacía"),
    ("cantidad_entregada", "NO_NUMERICA", "Valor no convertible a número"),
    ("cantidad_entregada", "NEGATIVA", "Cantidad menor que cero"),
    ("tiempo_entrega_prometido_hrs", "FALTANTE", "Tiempo vacío"),
    ("tiempo_entrega_prometido_hrs", "NO_NUMERICO", "Valor no convertible a número"),
    ("tiempo_entrega_prometido_hrs", "NO_POSITIVO", "Tiempo menor o igual que cero"),
    ("tiempo_entrega_real_hrs", "FALTANTE", "Tiempo vacío"),
    ("tiempo_entrega_real_hrs", "NO_NUMERICO", "Valor no convertible a número"),
    ("tiempo_entrega_real_hrs", "NEGATIVO", "Tiempo menor que cero"),
    ("otif_on_time", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
    ("otif_in_full", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
    ("otif", "FLAG_INVALIDA", "Valor distinto de 0 o 1"),
    ("tiempo_entrega_real_hrs+tiempo_entrega_prometido_hrs+otif_on_time", "INCONSISTENCIA_ON_TIME", "El flag no coincide con tiempo real <= tiempo prometido"),
    ("cantidad_entregada+cantidad_solicitada+otif_in_full", "INCONSISTENCIA_IN_FULL", "El flag no coincide con cantidad entregada >= cantidad solicitada"),
    ("otif_on_time+otif_in_full+otif", "INCONSISTENCIA_OTIF", "OTIF no coincide con la conjunción de on_time e in_full"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def detectar_identificador(df, columna, patron):
    valor_original = df[columna].astype("string")
    limpio = valor_original.str.strip()
    return [
        registrar_problema(df, limpio.eq(""), columna, "FALTANTE", valor_original),
        registrar_problema(df, limpio.ne("") & ~valor_original.str.fullmatch(patron).fillna(False), columna, "FORMATO_INVALIDO", valor_original),
    ]

def detectar_fechas_y_duplicados(df):
    fecha_original = df["fecha_despacho"].astype("string")
    fecha_limpia = fecha_original.str.strip()
    formato = fecha_original.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").fillna(False)
    fecha = pd.to_datetime(fecha_limpia, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    duplicada = texto(df, "order_id").duplicated(keep="first")
    return [
        registrar_problema(df, ~formato | fecha.isna(), "fecha_despacho", "FECHA_INVALIDA", fecha_original),
        registrar_problema(df, duplicada, "order_id", "DUPLICADO", df["order_id"]),
    ]

def detectar_centro(df):
    original = df["centro_distribucion"].astype("string")
    limpio = original.str.strip()
    permitidos = {"La Paz", "Cochabamba", "Santa Cruz", "Oruro", "Tarija"}
    return [
        registrar_problema(df, limpio.eq(""), "centro_distribucion", "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & ~original.isin(permitidos), "centro_distribucion", "VALOR_NO_RECONOCIDO", original),
    ]

def detectar_numero(df, columna, minimo, inclusivo=True, codigo_limite="NEGATIVO"):
    original = df[columna].astype("string")
    limpio = original.str.strip()
    numero = pd.to_numeric(limpio, errors="coerce")
    fuera = numero.lt(minimo) if inclusivo else numero.le(minimo)
    codigo_no_num = "NO_NUMERICA" if columna.startswith("cantidad") else "NO_NUMERICO"
    return [
        registrar_problema(df, limpio.eq(""), columna, "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & numero.isna(), columna, codigo_no_num, original),
        registrar_problema(df, numero.notna() & fuera, columna, codigo_limite, original),
    ]

def detectar_flags_y_coherencia(df):
    hallazgos = []
    numeros = {}
    for columna in ["otif_on_time", "otif_in_full", "otif"]:
        original = df[columna].astype("string")
        numeros[columna] = pd.to_numeric(original.str.strip(), errors="coerce")
        hallazgos.append(registrar_problema(df, ~numeros[columna].isin([0, 1]), columna, "FLAG_INVALIDA", original))
    solicitada = pd.to_numeric(texto(df, "cantidad_solicitada"), errors="coerce")
    entregada = pd.to_numeric(texto(df, "cantidad_entregada"), errors="coerce")
    prometido = pd.to_numeric(texto(df, "tiempo_entrega_prometido_hrs"), errors="coerce")
    real = pd.to_numeric(texto(df, "tiempo_entrega_real_hrs"), errors="coerce")
    base_tiempo = real.notna() & real.ge(0) & prometido.notna() & prometido.gt(0) & numeros["otif_on_time"].isin([0, 1])
    esperado_tiempo = real.le(prometido).astype("Int64")
    evidencia_tiempo = texto(df, "tiempo_entrega_real_hrs") + " <= " + texto(df, "tiempo_entrega_prometido_hrs") + "; flag=" + texto(df, "otif_on_time")
    hallazgos.append(registrar_problema(df, base_tiempo & numeros["otif_on_time"].ne(esperado_tiempo), "tiempo_entrega_real_hrs+tiempo_entrega_prometido_hrs+otif_on_time", "INCONSISTENCIA_ON_TIME", evidencia_tiempo))
    base_cantidad = solicitada.notna() & solicitada.gt(0) & entregada.notna() & entregada.ge(0) & numeros["otif_in_full"].isin([0, 1])
    esperado_cantidad = entregada.ge(solicitada).astype("Int64")
    evidencia_cantidad = texto(df, "cantidad_entregada") + " >= " + texto(df, "cantidad_solicitada") + "; flag=" + texto(df, "otif_in_full")
    hallazgos.append(registrar_problema(df, base_cantidad & numeros["otif_in_full"].ne(esperado_cantidad), "cantidad_entregada+cantidad_solicitada+otif_in_full", "INCONSISTENCIA_IN_FULL", evidencia_cantidad))
    base_otif = numeros["otif_on_time"].isin([0, 1]) & numeros["otif_in_full"].isin([0, 1]) & numeros["otif"].isin([0, 1])
    esperado_otif = (numeros["otif_on_time"] * numeros["otif_in_full"]).astype("Int64")
    evidencia_otif = "on_time=" + texto(df, "otif_on_time") + "; in_full=" + texto(df, "otif_in_full") + "; otif=" + texto(df, "otif")
    hallazgos.append(registrar_problema(df, base_otif & numeros["otif"].ne(esperado_otif), "otif_on_time+otif_in_full+otif", "INCONSISTENCIA_OTIF", evidencia_otif))
    return hallazgos

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = []
    for columna, patron in {
        "order_id": r"ORD-2026-\d{5}", "cliente_id": r"CLI-\d{3}",
        "producto_id": r"PROD-\d{3}", "camion_id": r"CAM-\d{2}", "chofer_id": r"CHO-\d{3}",
    }.items():
        hallazgos += detectar_identificador(principal, columna, patron)
    hallazgos += detectar_fechas_y_duplicados(principal)
    hallazgos += detectar_centro(principal)
    hallazgos += detectar_numero(principal, "cantidad_solicitada", 0, inclusivo=False, codigo_limite="NO_POSITIVA")
    hallazgos += detectar_numero(principal, "cantidad_entregada", 0, inclusivo=True, codigo_limite="NEGATIVA")
    hallazgos += detectar_numero(principal, "tiempo_entrega_prometido_hrs", 0, inclusivo=False, codigo_limite="NO_POSITIVO")
    hallazgos += detectar_numero(principal, "tiempo_entrega_real_hrs", 0, inclusivo=True, codigo_limite="NEGATIVO")
    hallazgos += detectar_flags_y_coherencia(principal)
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(lambda valores: "|".join(dict.fromkeys(valores)))
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())


## 4 · Reporte y comprobaciones antes de exportar

La huella SHA-256 identifica la versión exacta del CSV Bronze. Una fila puede acumular varios hallazgos, por lo que el total de problemas puede superar el número de filas en cuarentena.


In [ ]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name), ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO), ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales y luego reemplazan las salidas anteriores con los mismos nombres. El CSV Bronze nunca se sobrescribe.


In [ ]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_wms_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_wms_orders_diagnosticado.csv": df_diagnosticado,
    "andinalog_wms_orders_problemas.csv": df_problemas,
    "andinalog_wms_orders_cuarentena.csv": df_cuarentena,
    "andinalog_wms_orders_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. Ningún tratamiento queda aprobado por este diagnóstico. Una fila solo saldrá de cuarentena cuando todos sus problemas se hayan resuelto con reglas acordadas y comprobables.
